# Notebook 5.2: 2D vs 3D — Computational Scaling

## Objective
Compare 2D and 3D Stokes simulations and observe how cost scales with dimension.

**Key ideas:**
- 3D: extruded mesh, same physics, much larger system
- Cost scales as DOFs³ for direct solvers
- When is 2D sufficient? When does 3D matter?

In [ ]:
from fenics import *
import numpy as np
import matplotlib.pyplot as plt
import time

%matplotlib inline
set_log_level(LogLevel.WARNING)

---
## Part A: 2D Poiseuille (Reference)

In [ ]:
def solve_stokes_2d(nx, ny, L=1.0, H=0.1, mu=1.0, dp=1.0):
    mesh = RectangleMesh(Point(0.0, -H/2), Point(L, H/2), nx, ny)
    W = FunctionSpace(mesh, MixedElement([
        VectorElement("P", mesh.ufl_cell(), 2),
        FiniteElement("P", mesh.ufl_cell(), 1)
    ]))
    tol = 1e-10
    bc = DirichletBC(W.sub(0), Constant((0.0, 0.0)),
                     f"on_boundary && (x[1] > {H/2-tol} || x[1] < {-H/2+tol})")
    (u, p) = TrialFunctions(W)
    (v, q) = TestFunctions(W)
    a = (mu * inner(grad(u), grad(v)) - p*div(v) + q*div(u)) * dx
    f = Constant((-dp/L, 0.0))
    L_f = dot(f, v) * dx
    w = Function(W)
    t0 = time.time()
    solve(a == L_f, w, [bc])
    elapsed = time.time() - t0
    u_sol, _ = w.split()
    return W.dim(), elapsed, u_sol.vector().norm('linf')

# Run 2D for a sequence of mesh sizes
configs = [(20, 5), (40, 10), (80, 20)]
print(f"{'nx,ny':>12} {'DOFs':>8} {'Time (s)':>10} {'Max u':>10}")
print("-" * 45)
for nx, ny in configs:
    dofs, t, umax = solve_stokes_2d(nx, ny)
    print(f"({nx:>3},{ny:>3})     {dofs:>8}   {t:>8.3f}   {umax:>8.5f}")

---
## Part B: 3D Stokes (Extruded Channel)

In [ ]:
def solve_stokes_3d(nx, ny, nz, L=1.0, H=0.1, W3=0.1, mu=1.0, dp=1.0):
    mesh = BoxMesh(Point(0.0, -H/2, -W3/2), Point(L, H/2, W3/2), nx, ny, nz)
    W_fs = FunctionSpace(mesh, MixedElement([
        VectorElement("P", mesh.ufl_cell(), 2),
        FiniteElement("P", mesh.ufl_cell(), 1)
    ]))
    tol = 1e-10
    # No-slip on y-walls and z-walls
    def walls(x, on_boundary):
        return on_boundary and (
            abs(x[1] - H/2) < tol or abs(x[1] + H/2) < tol or
            abs(x[2] - W3/2) < tol or abs(x[2] + W3/2) < tol
        )
    bc = DirichletBC(W_fs.sub(0), Constant((0.0, 0.0, 0.0)), walls)
    (u, p) = TrialFunctions(W_fs)
    (v, q) = TestFunctions(W_fs)
    a = (mu * inner(grad(u), grad(v)) - p*div(v) + q*div(u)) * dx
    f = Constant((-dp/L, 0.0, 0.0))
    L_f = dot(f, v) * dx
    w = Function(W_fs)
    t0 = time.time()
    solve(a == L_f, w, [bc])
    elapsed = time.time() - t0
    u_sol, _ = w.split()
    return W_fs.dim(), elapsed, u_sol.vector().norm('linf')

# 3D with small mesh (coarse!)
print(f"{'nx,ny,nz':>14} {'DOFs':>8} {'Time (s)':>10} {'Max u':>10}")
print("-" * 50)
for (nx, ny, nz) in [(10, 4, 4), (20, 5, 5)]:
    dofs, t, umax = solve_stokes_3d(nx, ny, nz)
    print(f"({nx},{ny},{nz})         {dofs:>8}   {t:>8.3f}   {umax:>8.5f}")

---
## Part C: Scaling Summary

In [ ]:
print("Scaling Analysis:")
print()
print("2D: n elements per direction → n² total cells → n² DOFs")
print("3D: n elements per direction → n³ total cells → n³ DOFs")
print()
print("Direct solver (LU decomposition):")
print("  2D: time ∝ DOFs^2")
print("  3D: time ∝ DOFs^3")
print()
print("Rule of thumb: same relative mesh density in 3D takes ~100–1000x longer")
print()
print("When is 2D sufficient?")
print("  • If W (depth) >> H (height): Hele-Shaw limit, 2D is very accurate")
print("  • If aspect ratio is moderate: 3D gives full cross-section details")
print("  • Always validate 2D assumptions with a coarse 3D run first")

---
## Summary

| Dimension | Code change | Cost increase |
|-----------|------------|---------------|
| 2D → 3D | `RectangleMesh` → `BoxMesh`; velocity has 3 components | ~100–1000x |
| Aspect ratio W/H ≫ 1 | 2D Hele-Shaw is very accurate | Use 2D |
| Aspect ratio W/H ~ 1 | 3D needed for accurate corner flows | Use 3D |

**Exercise:** Compare the maximum velocity from 2D and 3D for a square cross-section (H = W). By how much do they differ?